In [0]:
# ----------------------------------------------------------
# Phase 4: Silver → Gold (Unify Batch + API + Stream)
# ----------------------------------------------------------

from pyspark.sql.functions import col, to_date, sum, count
import re

# ----------------------------------------------------------
# Step 0: Config - Storage Authentication
# ----------------------------------------------------------
storage_account_name = "stdataengportfolio"
storage_account_key  = "<STORAGE_ACCOUNT_KEY>"

silver_batch_path   = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/silver/superstore_historical/"
silver_api_path     = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/silver/superstore_api/"
silver_stream_path  = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/silver/superstore_stream/"
gold_path           = f"abfss://lake@{storage_account_name}.dfs.core.windows.net/gold/sales_aggregates/"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# ----------------------------------------------------------
# Helper: Clean column names
# ----------------------------------------------------------
def clean_columns(df):
    for c in df.columns:
        new_c = re.sub(r'[^a-zA-Z0-9_]', '', c.replace(' ', '_'))
        df = df.withColumnRenamed(c, new_c.lower())
    return df

# ----------------------------------------------------------
# Step 1: Load Silver Datasets
# ----------------------------------------------------------
print("➡️ Reading Silver layer data...")

df_batch  = clean_columns(spark.read.format("delta").load(silver_batch_path))
df_api    = clean_columns(spark.read.format("delta").load(silver_api_path))
df_stream = clean_columns(spark.read.format("delta").load(silver_stream_path))

print("✅ Silver data loaded.")

# ----------------------------------------------------------
# Step 2: Standardize Schema for Union
# ----------------------------------------------------------
# Rename sub_category consistently
for df in [df_batch, df_api, df_stream]:
    if "sub_category" in df.columns:
        df = df.withColumnRenamed("sub_category", "subcategory")

# Pick relevant columns
common_cols = [
    "order_id", "order_date", "category", "subcategory", 
    "product_id", "sales", "profit", "region"
]

df_batch  = df_batch.select(*[c for c in common_cols if c in df_batch.columns])
df_api    = df_api.select(*[c for c in common_cols if c in df_api.columns])
df_stream = df_stream.select(*[c for c in common_cols if c in df_stream.columns])

# ----------------------------------------------------------
# Step 3: Union All Data
# ----------------------------------------------------------
df_all = df_batch.unionByName(df_api, allowMissingColumns=True).unionByName(df_stream, allowMissingColumns=True)

print("✅ Unified dataset created.")

# ----------------------------------------------------------
# Step 4: Aggregations for Gold
# ----------------------------------------------------------
df_gold = (
    df_all.groupBy("order_date", "category", "subcategory", "region")
    .agg(
        sum("sales").alias("daily_sales"),
        sum("profit").alias("daily_profit"),
        count("order_id").alias("orders_count")
    )
    .orderBy("order_date")
)

print("✅ Gold aggregations complete.")

# ----------------------------------------------------------
# Step 5: Write to Gold Layer
# ----------------------------------------------------------
print(f"➡️ Writing to Gold: {gold_path}")
df_gold.write.format("delta").mode("overwrite").save(gold_path)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS gold_sales_aggregates
    USING DELTA
    LOCATION '{gold_path}'
""")

print("✅ Gold table ready.")


➡️ Reading Silver layer data...
✅ Silver data loaded.
✅ Unified dataset created.


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7130329421307374>, line 73
     67 print("✅ Unified dataset created.")
     69 # ----------------------------------------------------------
     70 # Step 4: Aggregations for Gold
     71 # ----------------------------------------------------------
     72 df_gold = (
---> 73     df_silver.groupBy("order_date", "category", "subcategory", "region", "product_name")
     74     .agg(
     75         sum("total_sales").alias("daily_sales"),
     76         sum("total_profit").alias("daily_profit"),
     77         count("order_id").alias("orders_count")
     78     )
     79     .orderBy("order_date")
     80 )
     83 print("✅ Gold aggregations complete.")
     85 # ----------------------------------------------------------
     86 # Step 5: Write to Gold Layer
     87 # -------------------------------------------------------